In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


This study was conducted to identify the reason why the expected gene–gene correlations were not being preserved. In particular, I examined whether the reverse gene expression reconstruction method was responsible for the loss of correlation structure.

For PBMC

In [ ]:
# ============================================================
# DIAGNOSTIC: Reverse REAL genomaps → check correlation recovery
# ============================================================

import numpy as np
from scipy.stats import pearsonr, spearmanr

BASE      = '/content/drive/MyDrive/Ahsan/8. 2025 cGAN Hamid with Dr. Darren Dataset'
REV_DOCS  = BASE + '/Reverse GeneExpression Required Docs'

# Load real genomaps and real gene expression
b_real    = np.load(BASE + '/b_Class_genomap.npy').astype(np.float32)
mono_real = np.load(BASE + '/mono_Class_genomap.npy').astype(np.float32)
if b_real.ndim == 3:    b_real    = b_real[..., np.newaxis]
if mono_real.ndim == 3: mono_real = mono_real[..., np.newaxis]

import pandas as pd
real_b_expr    = pd.read_csv('/content/drive/MyDrive/Ahsan/16. Review SynCellNet  work/Dataset/Real PBMC dataset/b_Class_dataset.csv',    header=None).values.astype(np.float32)
real_mono_expr = pd.read_csv('/content/drive/MyDrive/Ahsan/16. Review SynCellNet  work/Dataset/Real PBMC dataset/mono_Class_dataset.csv', header=None).values.astype(np.float32)

T_b    = np.load(REV_DOCS + '/transformation_matrix_T_b_class.npy')
T_mono = np.load(REV_DOCS + '/transformation_matrix_T_mono_class.npy')
means_b    = np.load(REV_DOCS + '/original_means_b_class.npy')
means_mono = np.load(REV_DOCS + '/original_means_mono_class.npy')
stds_b     = np.load(REV_DOCS + '/original_stds_b_class.npy')
stds_mono  = np.load(REV_DOCS + '/original_stds_mono_class.npy')

num_genes      = T_b.shape[1]
totalGridPoint = 40 * 40

# Recompute _MIN/_MAX from real data
stacked = np.vstack([b_real, mono_real])
_MIN = float(stacked.min())
_MAX = float(stacked.max())
print(f"_MIN={_MIN:.4f}, _MAX={_MAX:.4f}")

# ---- Option B reverse on REAL genomaps ----
def reverse_optionB(genomaps_original_scale, T, means, stds):
    n = genomaps_original_scale.shape[0]
    projMat     = T * totalGridPoint
    projMat_inv = np.linalg.pinv(projMat)
    projM = np.zeros((n, num_genes), dtype=np.float32)
    for i in range(n):
        ex = genomaps_original_scale[i, :, :, 0].flatten(order='F')
        projM[i, :] = ex[:num_genes]
    X_norm = np.matmul(projM, projMat_inv)
    return (X_norm * stds) + means

# Real genomaps are already in original scale — no min-max reversal needed
recovered_b_real    = reverse_optionB(b_real,    T_b,    means_b,    stds_b)
recovered_mono_real = reverse_optionB(mono_real, T_mono, means_mono, stds_mono)

# ---- Compare correlation matrices ----
n_top = 100
top_var_idx = np.argsort(real_b_expr.var(axis=0))[-n_top:]

corr_real_b       = np.corrcoef(real_b_expr[:, top_var_idx].T)
corr_recovered_b  = np.corrcoef(recovered_b_real[:, top_var_idx].T)
corr_real_mono    = np.corrcoef(real_mono_expr[:, top_var_idx].T)
corr_recovered_mono = np.corrcoef(recovered_mono_real[:, top_var_idx].T)

triu = np.triu_indices(n_top, k=1)
r_b,    _ = pearsonr(corr_real_b[triu],    corr_recovered_b[triu])
r_mono, _ = pearsonr(corr_real_mono[triu], corr_recovered_mono[triu])

print(f"\n--- DIAGNOSTIC RESULT ---")
print(f"Pearson r of correlation matrices (real vs recovered-from-real-genomap):")
print(f"  B class:    {r_b:.4f}")
print(f"  Mono class: {r_mono:.4f}")
print(f"\nInterpretation:")
print(f"  r ≈ 1.0  → reverse transformation is correct, cGAN is the problem")
print(f"  r ≈ 0.0  → reverse transformation itself destroys correlation")

_MIN=-2.4650, _MAX=34.4093

--- DIAGNOSTIC RESULT ---
Pearson r of correlation matrices (real vs recovered-from-real-genomap):
  B class:    1.0000
  Mono class: 0.9999

Interpretation:
  r ≈ 1.0  → reverse transformation is correct, cGAN is the problem
  r ≈ 0.0  → reverse transformation itself destroys correlation


For PDO

In [ ]:
# ============================================================
# DIAGNOSTIC: Reverse REAL genomaps → check correlation recovery
# PDO DATASET
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import pearsonr

BASE     = '/content/drive/MyDrive/Ahsan/11. Cancer Data on 30th April'
REV_DOCS = BASE + '/Reverse Calculation Related doc'

# Load real genomaps
low_real  = np.load(BASE + '/Real_Low_genomap.npy').astype(np.float32)
high_real = np.load(BASE + '/Real_High_genomap.npy').astype(np.float32)
if low_real.ndim == 3:  low_real  = low_real[...,  np.newaxis]
if high_real.ndim == 3: high_real = high_real[..., np.newaxis]
print(f"Low genomap: {low_real.shape} | High genomap: {high_real.shape}")

# Load real gene expression (raw)
real_low_expr  = pd.read_csv(BASE + '/3. Differential_Low_Raw_Finalized.csv',  header=None).values.astype(np.float32)
real_high_expr = pd.read_csv(BASE + '/3. Stem_High_Raw_Finalized.csv', header=None).values.astype(np.float32)
print(f"Low expr: {real_low_expr.shape} | High expr: {real_high_expr.shape}")

# Load T matrices and stats
T_low  = np.load(REV_DOCS + '/transformation_matrix_T_Low_class.npy')
T_high = np.load(REV_DOCS + '/transformation_matrix_T_High_class.npy')
means_low  = np.load(REV_DOCS + '/original_means_Low_class.npy')
means_high = np.load(REV_DOCS + '/original_means_High_class.npy')
stds_low   = np.load(REV_DOCS + '/original_stds_Low_class.npy')
stds_high  = np.load(REV_DOCS + '/original_stds_High_class.npy')

print(f"T_low shape: {T_low.shape} | T_high shape: {T_high.shape}")
num_genes      = T_low.shape[1]
totalGridPoint = 40 * 40

# Recompute _MIN/_MAX from real data (same as training)
stacked = np.vstack([low_real, high_real])
_MIN = float(stacked.min())
_MAX = float(stacked.max())
print(f"_MIN={_MIN:.6f}, _MAX={_MAX:.6f}")

# ---- Option B reverse on REAL genomaps ----
def reverse_optionB(genomaps_original_scale, T, means, stds):
    n = genomaps_original_scale.shape[0]
    projMat     = T * totalGridPoint
    projMat_inv = np.linalg.pinv(projMat)
    projM = np.zeros((n, num_genes), dtype=np.float32)
    for i in range(n):
        ex = genomaps_original_scale[i, :, :, 0].flatten(order='F')
        projM[i, :] = ex[:num_genes]
    X_norm = np.matmul(projM, projMat_inv)
    return (X_norm * stds) + means

# Real genomaps are already in original scale — no min-max reversal needed
recovered_low  = reverse_optionB(low_real,  T_low,  means_low,  stds_low)
recovered_high = reverse_optionB(high_real, T_high, means_high, stds_high)

# ---- Compare correlation matrices ----
n_top = 100
top_var_idx = np.argsort(real_low_expr.var(axis=0))[-n_top:]

corr_real_low       = np.corrcoef(real_low_expr[:,  top_var_idx].T)
corr_recovered_low  = np.corrcoef(recovered_low[:,  top_var_idx].T)
corr_real_high      = np.corrcoef(real_high_expr[:, top_var_idx].T)
corr_recovered_high = np.corrcoef(recovered_high[:, top_var_idx].T)

triu = np.triu_indices(n_top, k=1)
r_low,  _ = pearsonr(corr_real_low[triu],  corr_recovered_low[triu])
r_high, _ = pearsonr(corr_real_high[triu], corr_recovered_high[triu])

print(f"\n--- PDO DIAGNOSTIC RESULT ---")
print(f"Pearson r of correlation matrices (real vs recovered-from-real-genomap):")
print(f"  Low (Differentiated) class: {r_low:.4f}")
print(f"  High (Stem) class:          {r_high:.4f}")
print(f"\nInterpretation:")
print(f"  r ≈ 1.0  → reverse transformation is correct, cGAN is the problem")
print(f"  r ≈ 0.0  → reverse transformation itself destroys correlation")

Low genomap: (1415, 40, 40, 1) | High genomap: (1415, 40, 40, 1)
Low expr: (1415, 1600) | High expr: (1415, 1600)
T_low shape: (1600, 1600) | T_high shape: (1600, 1600)
_MIN=-0.738665, _MAX=37.589901

--- PDO DIAGNOSTIC RESULT ---
Pearson r of correlation matrices (real vs recovered-from-real-genomap):
  Low (Differentiated) class: 1.0000
  High (Stem) class:          0.9996

Interpretation:
  r ≈ 1.0  → reverse transformation is correct, cGAN is the problem
  r ≈ 0.0  → reverse transformation itself destroys correlation


In [ ]:
# Sanity check: compare actual values, not just correlation structure
mae_low  = np.mean(np.abs(recovered_low  - real_low_expr))
mae_high = np.mean(np.abs(recovered_high - real_high_expr))
print(f"Mean Absolute Error — Low:  {mae_low:.4f}")
print(f"Mean Absolute Error — High: {mae_high:.4f}")
# If MAE ≈ 0 → code is correct, reverse is genuinely lossless
# If MAE is large → code is computing correlation of something with itself

Mean Absolute Error — Low:  0.0000
Mean Absolute Error — High: 0.0001
